# Using the Analytics Engine (AE) to produce heating and cooling degree days
This notebook aims to reproduce the workflow CEC's Demand Forecast Unit takes to generate weather and climate information for the annual consumption model. Here the existing workflow is replicated, but connecting with new data from California's Fifth Climate Change Assessment.

To execute a given 'cell' of this notebook, place the cursor in the cell and press the 'play' icon, or simply press shift+enter together. Some cells will take longer to run, and you will see a [$\ast$] to the left of the cell while AE is still working.

**Intended Application**: As a user, I want to **<span style="color:#FF0000">generate heating and cooling degree days</span>** as input for an annual energy consumption model by:
1. Understand and visualize the difference between data at the weather station, and aggregated across the demand zone
2. Compute and visualize trends in heating and cooling degree days with a flexible threshold
3. Compute and visualize trends in heating and cooling degree hours with a flexible threshold

**Runtime**: With the default settings, this notebook takes approximately **11 minutes** to run from start to finish. Modifications to selections may increase the runtime. 

## Step 0: Setup
First, we'll import any general python libraries required to run the notebook. We'll also import the python library climakitae, our AE toolkit for climate data analysis, along with specific functions that we'll use within this notebook. 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import hvplot.pandas

import climakitae as ck
import climakitaegui as ckg
from climakitae.core.data_interface import get_data
from climakitae.util.utils import (
    compute_annual_aggreggate,
    trendline,
    compute_multimodel_stats,
    combine_hdd_cdd,
)
from climakitae.tools.derived_variables import compute_hdd_cdd, compute_hdh_cdh
from climakitaegui.util.utils import hdd_cdd_lineplot, hdh_cdh_lineplot

from dask.diagnostics import ProgressBar

## Step 1: Working with single gridpoint bias-adjusted to station
This first example shows how to calculate HDD/CDD with bias-adjusted data. It shows how to replicate the historical observations at Sacramento Executive Airport by grabbing the grid cell from the model nearest to the airport. It is **critical** to note that the station-based grid cell data we are retrieving is **bias-adjusted**. In the example in Step 2, the gridded data that is retrieved is **not bias-adjusted** and therefore should be carefully considered.

### 1a) Read in the data 
Here, we'll use the `get_data` function to load the data.

Becasuse the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezome of the station we've selected. This is particularly important for determining the timing of the daily maximum and minimum temperatures. For a station located in Pacific Time (US), UTC time places the daily minimum "in" the day prior because UTC is 8 hours ahead of Pacific! 

In [ ]:
## set station name
station_name = "KSAC"

# select historical data
cd = ck.ClimateData(verbosity=0)
data_at_station = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d02")
    .variable("t2")
    .processes(
        {
            "convert_to_local_time": {"convert": "yes", "reindex_time_axis": "no"},
            "convert_units": "degF",
            "time_slice": (2005, 2025),
            "bias_adjust_model_to_station": {
                "stations": [station_name],
                "historical_slice": (
                    1980,
                    2014,
                ),  # You can specify what historical period to bias correct for.
            },
        }
    )
).get()

### 1b) Load the data into memory
This may take some time, because the data has to be loaded into memory and then subsetted to get the closest grid cell. All computations we've done before this step are actually computed in this step; before, we just see a preview of the data. Because of this, **we recommend running this notebook in the Analytics Engine's Jupyter Hub, which provides additional computational resources that greatly speed up this step.**

In [ ]:
with ProgressBar():
    data_at_station = data_at_station["Sacramento Executive Airport (KSAC)"].compute()

### 1c) Output air temperature product as a csv file
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. In the output table, the first column is the local time, and the second column are the various global climate models (which can be filtered in excel or in python code in the notebook). The other columns are the variables selected at the beginning of the notebook.

In [ ]:
data_at_station_df = data_at_station.to_dataframe()
data_at_station_df.head()

In [ ]:
filename = "hourly_data_at_station_{0}.csv".format(
    station_name.replace(" ", "_")
).lower()
data_at_station_df.to_csv(filename, index=True)

### 1d) Compute heating degree days and cooling degree days
Degree days are [based on the assumption](https://www.weather.gov/key/climate_heat_cool) that when the outside temperature is 65°F, we don't need heating or cooling to be comfortable. However, you may wish to use a different threshold than 65°F: for example, you might want to assume that temperatures between 60-70°F won't require heating or cooling, and use degrees below 60°F for HDD and degrees above 70°F for CDD. <br><br> In the code below, a heating degree day (HDD) is calculated by computing how many degrees Fahrenheit **colder** the daily temperature is from a specified temperature threshold. A cooling degree day (CDD) is calculated by computing how many degrees **warmer** the daily temperature is from a specified temperature threshold. In the computation below, you can provide different thresholds for HDD and CDD based on your needs. 

#### 1d.i) Compute HDD and CDD 
We'll use the climakitae helper function `compute_hdd_cdd` to compute both heating and cooling degree days, which uses the function arguments `hdd_threshold` and `cdd_threshold` to represent any threshold of your choosing. In the example below, we will calculate HDD with a threshold of 65 degF and CDD with a threshold of 65 degF. The function performs the following calculations:<br><br>
**HDD = threshold - temperature<br>
CDD = (-1)\*(threshold - temperature)**<br><br>
For HDD, we can just subtract the 2m temperature from the selected threshold, then set any negative value to 0. For CDD, we will do the same, but will then multiply by -1 to turn negative values to positive, then set negative values to 0. We need to multiply by -1 for CDD to avoid having all negative values; for example, on a day of 80F and a cdd_threshold of 70F, CDD = 70 - 80 = -10, but the CDD value is +10. Multiplying -10 by -1 will give us the true value of 10.

In [ ]:
# help(compute_hdd_cdd) # See information about the function

In [ ]:
# Resample to daily mean
data_to_use = data_at_station.resample(time="1D").mean().rename("t2")

# Change the temperature thresholds here
hdd, cdd = compute_hdd_cdd(data_to_use, hdd_threshold=65, cdd_threshold=65)

#### 1d.ii) Aggregate annually to find HDD and CDD per year

To do this, we will first group the data by year and compute a sum across space and time. Then, we will divide the annual aggregated data by the number of grid cells over which the sum was computed.

In [ ]:
hdd_annual_station = compute_annual_aggreggate(
    data=hdd, name="Annual Heating Degree Days (HDD)", num_grid_cells=1
)
cdd_annual_station = compute_annual_aggreggate(
    data=cdd, name="Annual Cooling Degree Days (CDD)", num_grid_cells=1
)

#### 1d.iii) Compute the multimodel mean, min, and max. 
We'll add these statistics to our main datasets, `hdd_annual` and `cdd_annual`, so they can be easily accessed for plotting.

In [ ]:
hdd_annual_station = compute_multimodel_stats(hdd_annual_station)
cdd_annual_station = compute_multimodel_stats(cdd_annual_station)

## 1e) Compute heating degree hours and cooling degree hours
Alternatively, you may be interested in the number of hours in each day that a designated heating or cooling threshold crosses. For Cooling Degree Hours (CDH), this is the number of hours in which the hourly temperature exceeds the cooling degree threshold. Likewise, Heating Degree Hours (HDH) is the number of hours in which the hourly temperature is below the heating degree threshold. We'll use the helper function `compute_hdh_cdh` to calculate HDH and CDH:<br><br>
**CDH = num of hours where (temperature $>$ threshold)<br>
HDH = num of hours where (temperature $<$ threshold)**<br><br>
We will display the results to see how trends change throughout the year. 

#### 1e.i) Compute HDH and CDH
Like the CDD and HDD examples above, we'll use all of the data for our selected DFZ zone to calculate CDH and HDH. Note that we've added an attribute to the data to retain the threshold used to compute the data here. If you forget, look at the attributes of CDH or HDH. 

In [ ]:
# help(compute_hdh_cdh) # See information about the function

In [ ]:
data_to_use = data_at_station.rename("t2")  # reset to hourly data
hdh, cdh = compute_hdh_cdh(data_to_use, hdh_threshold=65, cdh_threshold=75)

#### 1e.ii) Display a month of CDH and HDH
Next, we'll plot specific months of the overall timeseries produced by the CDH and HDH calculation to see the trend in degree hours. We'll use a helper plotting function, and input  a month of interest. For example, we'll look at June of 2011, but you can input any date of interest; we provide examples for plotting a specific month and a specific year below.

In [ ]:
data_one_month = cdh.sel(time="June 2011")
hdh_cdh_lineplot(data_one_month)

In [ ]:
data_one_month = hdh.sel(time="June 2011")
hdh_cdh_lineplot(data_one_month)

Alternatively, it may be useful to visualize a specific year to see the trends over time. We'll do this for 2021 as an example below with Cooling Degree Hours. 

In [ ]:
data_one_year = cdh.sel(time="2021")
hdh_cdh_lineplot(data_one_year)

#### 1e.iii) Output data as csv files
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. 

In [ ]:
# Merge and simplify data
hdh_cdh_combined = xr.merge([combine_hdd_cdd(hdh), combine_hdd_cdd(cdh)])
hdh_cdh_combined = ck.load(hdh_cdh_combined)

# Convert to pandas dataframe
hdh_cdh_df = hdh_cdh_combined.to_dataframe()
hdh_cdh_df.head()

In [ ]:
filename = "daily_hdh_cdh_{0}.csv".format(station_name.replace(" ", "_").lower())
hdh_cdh_df.to_csv(filename, index=True)

## Step 2: Working with gridded data and derived HDD/CDD

As an alternative to a single point, we can instead consider weather conditions across an entire forecast zone. In this example, we calculate the median of all conditions across the Sacramento Municipal Utility District. We have pre-loaded data selections for gridded HDD and CDD within the Sacramento Municipal Utility District for 2005-2025. The HDD and CDD derived variables use a threshold of **65°F** for both metrics. Feel free to make modifications for your own workflows. <br>
**Reminder**: The gridded data we will be retrieving in this step and using throughout this notebook is not bias-corrected.

In [ ]:
cd = ck.ClimateData(verbosity=0)
hdd_dfz_aggregated = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .variable("HDD_wrf")
    .processes(
        {
            "convert_to_local_time": {"convert": "yes", "reindex_time_axis": "no"},
            "time_slice": (2005, 2025),
            "clip": "SMUD Service Territory",
            # Can aggregate with median, mean, max, or min
            "metric_calc": {"metric": "median", "dims": ["x", "y"]},
        }
    )
).get()

cdd_dfz_aggregated = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .variable("CDD_wrf")
    .processes(
        {
            "convert_to_local_time": {"convert": "yes", "reindex_time_axis": "no"},
            "time_slice": (2005, 2025),
            "clip": "SMUD Service Territory",
            # Can aggregate with median, mean, max, or min
            "metric_calc": {"metric": "median", "dims": ["x", "y"]},
        }
    )
).get()

### 2b) Load the data into memory
The data is loaded into memory before further computations. This may take several minutes.

In [ ]:
with ProgressBar():
    hdd_dfz_aggregated = hdd_dfz_aggregated["HDD_wrf"].compute()
    cdd_dfz_aggregated = cdd_dfz_aggregated["CDD_wrf"].compute()

In [ ]:
hdd_dfz_aggregated

### 2b) Aggregate annually to find HDD and CDD per year

To do this, we will first group the data by year and compute a sum across space and time. Then, we will divide the annual aggregated data by the number of grid cells over which the sum was computed.

In [ ]:
hdd_annual_dfz = compute_annual_aggreggate(
    data=hdd_dfz_aggregated, name="Annual Heating Degree Days (HDD)", num_grid_cells=1
)
cdd_annual_dfz = compute_annual_aggreggate(
    data=cdd_dfz_aggregated, name="Annual Cooling Degree Days (CDD)", num_grid_cells=1
)

### 2c) Compute the multimodel mean, min, and max. 
We'll add these statistics to our main datasets, `hdd_annual` and `cdd_annual`, so they can be easily accessed for plotting.

In [ ]:
hdd_annual_dfz = compute_multimodel_stats(hdd_annual_dfz)
cdd_annual_dfz = compute_multimodel_stats(cdd_annual_dfz)

## Step 3: Compute the median value of the grid cells in station's corresponding forecast zone

In this example, we will visualize the data across the Demand Forecast Zone for the Sacramento Municipal Utility District, and then calculate the median of all conditions across the Sacramento Municipal Utility District.

# Visualization

### 4e) Compute a trendline using the mean of all simulations
We'll find the coefficients for a first degree (linear) polynomial using [numpy's `polyfit` function](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). The returned coefficients (**m** and **b** in the code below) will allow us to compute the trendline using the linear polynomial y = mx + b, where **y** is the trendline and **x** is the years. 

In [ ]:
# Choose the station data
# hdd_annual = hdd_annual_station
# cdd_annual = cdd_annual_station

# OR choose the DFZ data
hdd_annual = hdd_annual_dfz
cdd_annual = cdd_annual_dfz

# Get trendline fit
hdd_trendline = trendline(hdd_annual, kind="mean")
cdd_trendline = trendline(cdd_annual, kind="mean")

### 4f) Visualize the results
Using the python package *hvplot*, we can easily make a line plot of the annual aggregated data. To do this, we'll plot the annual HDD, then add the trendline on top. The code to generate the plot is contained in a function `hdd_cdd_lineplot`. 

Please note, the gridded data is not currently bias-corrected. As a result of this, the minimum or maximum timeseries could reflect a single simulation that is biased high or low compared to others. You can toggle lines on and off in the plots below by clicking on the name in the legend. 

In [ ]:
hdd_cdd_lineplot(
    annual_data=hdd_annual,
    trendline=hdd_trendline,
    title="Annual Aggregate Heating Degree Days",
)

In [ ]:
hdd_cdd_lineplot(
    annual_data=cdd_annual,
    trendline=cdd_trendline,
    title="Annual Aggregate Cooling Degree Days",
)

### 4g) Output data as csv files
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. 

In [ ]:
# Merge and simplify data
hdd_cdd_combined = xr.merge([combine_hdd_cdd(hdd_annual), combine_hdd_cdd(cdd_annual)])
hdd_cdd_combined = ck.load(hdd_cdd_combined)

# Convert to pandas dataframe
hdd_cdd_df = hdd_cdd_combined.to_dataframe()
hdd_cdd_df.head()

In [ ]:
filename = "annual_hdd_cdd_{0}.csv".format(station_name.replace(" ", "_").lower())
hdd_cdd_df.to_csv(filename, index=True)